In [1]:
import os
import base64
import json
import random
from openai import OpenAI
import anthropic
import numpy as np
import re
from tqdm import tqdm
import pandas as pd
import time
from word2number import w2n
from dotenv import load_dotenv


# Load dataset

In [ ]:
# Set base directory using relative path
base_dir = os.path.join(os.getcwd(), "dataset", "simpsons")

# Set paths relative to base_dir
annotation_path = os.path.join(base_dir, "v1_Annotation_Val_simpsons_vqa.json")
question_path = os.path.join(base_dir, "v1_Question_Val_simpsons_vqa.json")
images_dir = os.path.join(base_dir, "val_images")

def load_dataset(annotation_path, question_path):
    try:
        with open(annotation_path, 'r') as f:
            annotations = json.load(f)['annotations']

        with open(question_path, 'r') as f:
            questions = json.load(f)['questions']

        # Select high-quality QA pairs (overall_scores == 1.0)
        filtered_annotations = [
            annotation for annotation in annotations
            if annotation.get('overall_scores', {}).get('question') == 1.0 and
               annotation.get('overall_scores', {}).get('answer') == 1.0
        ]

        # Create a mapping from question ID to answer
        question_id_to_answer = {
            annotation['id']: annotation['answer']
            for annotation in filtered_annotations
        }

        # Create a mapping from question ID to answer type
        question_id_to_answer_type = {
            annotation['id']: {
                'answer': annotation['answer'],
                'answer_type': annotation.get('answer_type', 'other')
            }
            for annotation in filtered_annotations
        }

        filtered_questions = [question for question in questions if question['id'] in question_id_to_answer]

        return filtered_questions, filtered_annotations, question_id_to_answer, question_id_to_answer_type

    except Exception as e:
        print(f"Error loading dataset: {e}")
        return [], [], {}, {}

def get_dataset(questions, question_id_to_answer, fraction=0.005, seed=42):
    # TODO：Increase quantity
# def get_dataset(questions, question_id_to_answer, fraction=0.05, seed=42):
    try:
        random.seed(seed)
        sample_size = max(1, int(len(questions) * fraction))
        sampled_questions = random.sample(questions, sample_size)
        sampled_correct_answers = [
            question_id_to_answer[q['id']]
            for q in sampled_questions
        ]
        # Add image_base64 to each sampled question
        for q in sampled_questions:
            image_relative_path = q['img_path']
            image_path = os.path.join(images_dir, image_relative_path)
            q['image_base64'] = encode_image(image_path)

        return sampled_questions, sampled_correct_answers

    except Exception as e:
        print(f"Error sampling dataset: {e}")
        return [], []

def encode_image(image_path):
    try:
        if not os.path.exists(image_path):
            print(f"Error: The image file at {image_path} was not found.")
            return None

        with open(image_path, "rb") as image_file:
            image_base64 = base64.b64encode(image_file.read()).decode('utf-8')
            return image_base64

    except Exception as e:
        print(f"An error occurred while encoding the image: {e}")
        return None
    
# Initialize results list
results_standard = []

# Multi agent

In [ ]:
load_dotenv()
# Configuration
MODEL_NAME = "claude-3-5-haiku-20241022"
# MODEL_NAME = "gpt-4o-mini"

is_openai_model = not MODEL_NAME.startswith("claude-")

if is_openai_model:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    print(f"Using OpenAI model: {MODEL_NAME}")
else:
    client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
    print(f"Using Anthropic model: {MODEL_NAME}")

# Visual agent: handles image-related tasks，and outputs image description
def visual_agent(image_base64, question, max_retries=3, retry_delay=2):
    prompt = f"""
    As a visual analysis expert, carefully analyze the image and provide a concise visual description. Avoid speculation or assumptions beyond the visible content.
    Focus on the following aspects, as relevant to answering the question: {question}.

    1. Characters: Identify characters with distinctive features.
    2. Actions and Interactions: Describe what character is doing, including body posture and interactions.
    3. Facial Expressions and Emotions: Note visible facial expressions (e.g., happy, surprised, angry).
    4. Scene: Identify whether the scene is indoors or outdoors, and specify the environment.
    5. Objects: Mention relevant items, positions, colors, and sizes.
    6. Layout: Describe where characters and objects are located (e.g., left of, behind).
    7. Attributes and Colors: List visible colors and give exact counts where possible.
    8. Counts: Number of characters or repeated items.
    9. Movement: Describe motion or visual cues if any.
    """

    for attempt in range(max_retries):
        try:
            if is_openai_model:
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[
                        {
                            "role": "user",
                            "content": [
                                {"type": "text", "text": prompt},
                                {"type": "image_url",
                                 "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}},
                            ],
                        }
                    ],
                    max_tokens=1000,
                    temperature=0.1,
                )
                visual_desc = completion.choices[0].message.content.strip()
                return visual_desc
            else:
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image",
                             "source": {"type": "base64", "media_type": "image/jpeg", "data": image_base64}},
                        ]
                    }],
                    max_tokens=1000,
                    temperature=0.1,
                )
                visual_desc = completion.content[0].text.strip()
                return visual_desc

        except Exception as e:
            print(f"Visual agent attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            continue

    print("Error: Visual agent failed to process the image")
    return None

# Language agent: handles text-related tasks,and outputs initial predicted answer
def language_agent(question, image_base64, visual_desc, max_retries=3, retry_delay=2):
    prompt = f"""
    As a cartoon language expert, answer the question based on the provided context using EXACTLY ONE WORD:

    Input:
    Image Description: {visual_desc}
    Question: {question}

    Guidelines: No explanations or punctuation allowed.
    """

    for attempt in range(max_retries):
        try:
            if is_openai_model:
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[
                        {
                            "role": "user",
                            "content": [
                                {"type": "text", "text": prompt},
                                {
                                    "type": "image_url",
                                    "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
                                }
                            ],
                        }
                    ],
                    max_tokens=10,
                    temperature=0.1,
                )
                initial_predicted_answer = completion.choices[0].message.content.strip().lower()
            else:
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image",
                             "source":
                                 {"type": "base64", "media_type": "image/jpeg", "data": image_base64}},
                        ]
                    }],
                    max_tokens=10,
                    temperature=0.1,
                )
                initial_predicted_answer = completion.content[0].text.strip().lower()

            # Process the response
            # 1. Remove punctuation marks
            initial_predicted_answer = initial_predicted_answer.rstrip('.!?')

            # 2. Split into words and get the first word
            words = initial_predicted_answer.split()
            if not words:
                continue

            initial_predicted_answer = words[0]

            # 3. Convert numbers if applicable
            try:
                # Check if word represents a number
                number = w2n.word_to_num(initial_predicted_answer)
                initial_predicted_answer = str(number)
            except ValueError:
                # Check if contains numeric digits
                matches = re.findall(r'\d+', initial_predicted_answer)
                if matches:
                    initial_predicted_answer = matches[0]

            return initial_predicted_answer

        except Exception as e:
            print(f"Language agent attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            continue

    # If all attempts failed, provide a simple answer based on the question type
    print("Error: Language agent failed to generate an answer")
    return None

# Hallucination detection agent: detect hallucinations by comparing language agent's answer with Correct Answer
def hallucination_agent(question, image_base64, initial_predicted_answer, visual_desc, max_retries=3, retry_delay=2):
    prompt = f"""
    As a cartoon hallucination detection expert, verify whether the predicted answer is accurate based on the available information.

    Input:
    Question: {question}
    Image Context: {visual_desc}
    Predicted Answer: {initial_predicted_answer}

    Guidelines:
    1. Accuracy: Verify if the prediction is consistent with the image.
    2. Support: Check if the visual evidence supports the prediction.
    3. Completeness: Ensure the answer contains the key information needed to answer the question.
    4. Error Analysis: If inaccuracies exist, identify what specific information was misunderstood or overlooked.
    5. Self-reflection: Consider why the model might have produced these inaccuracies (e.g., misleading visual cues, ambiguity).
    6. You MUST respond in ONE of these two formats ONLY:
       KEEP: [original answer] - if the prediction is accurate or you're uncertain
       REVISE: [one-word corrected answer] - if the prediction needs revision.
    7. Answer format considerations based on question type:
       - For yes/no questions (starting with "is", "are", "does", etc.): Ensure the answer is either "yes" or "no"
       - For number questions (starting with "how many"): Ensure the answer is a numeric value only
       - For other questions: Provide the most concise accurate answer
    """

    for attempt in range(max_retries):
        try:
            if is_openai_model:
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image_url",
                             "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}}
                        ]
                    }],
                    max_tokens=150,
                    temperature=0.1,
                )
                response = completion.choices[0].message.content.strip()
            else:
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image",
                             "source": {
                                 "type": "base64",
                                 "media_type": "image/jpeg",
                                 "data": image_base64
                             }}
                        ]
                    }],
                    max_tokens=150,
                    temperature=0.1,
                )
                response = completion.content[0].text.strip()

            # Process the response
            # First attempt to match standard format
            keep_match = re.match(r'^KEEP:\s*(\w+)$', response, re.IGNORECASE)
            revise_match = re.match(r'^REVISE:\s*(\w+)$', response, re.IGNORECASE)

            if keep_match:
                final_answer = initial_predicted_answer.lower()
            elif revise_match:
                # Retrieve the corrected answer
                final_answer = revise_match.group(1).lower()
                # Remove punctuation marks
                final_answer = final_answer.rstrip('.!?')
                # Convert numerical words to digits
                try:
                    number = w2n.word_to_num(final_answer)
                    final_answer = str(number)
                except ValueError:
                    matches = re.findall(r'\d+', final_answer)
                    if matches:
                        final_answer = matches[0]
            else:
                # If format doesn't match, attempt a more flexible extraction approach
                if response.lower().startswith("keep"):
                    final_answer = initial_predicted_answer.lower()
                elif response.lower().startswith("revise"):
                    # Extract content following "REVISE"
                    content = response.split(":", 1)[1] if ":" in response else response.replace("REVISE", "", 1)
                    # Process consistently with language_agent
                    content = content.strip().lower().rstrip('.!?')
                    words = content.split()
                    if words:
                        # Extract only the first word
                        final_answer = words[0]
                        # Convert numerical words to digits
                        try:
                            number = w2n.word_to_num(final_answer)
                            final_answer = str(number)
                        except ValueError:
                            matches = re.findall(r'\d+', final_answer)
                            if matches:
                                final_answer = matches[0]
                    else:
                        final_answer = initial_predicted_answer.lower()
                else:
                    # Default to using the original answer
                    final_answer = initial_predicted_answer.lower()

            return final_answer

        except Exception as e:
            print(f"Hallucination check attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            continue

    # If all retry attempts fail, return the initial prediction
    return initial_predicted_answer.lower()


# Calculate accuracy

In [ ]:
def compute_accuracy(question, correct_answer, predicted_answer, answer_type, max_retries=2, retry_delay=2, num_evaluations=3):
    if correct_answer.lower().strip() == predicted_answer.lower().strip():
        return 1.0, [1.0] * num_evaluations

    if correct_answer.lower().strip() + 's' == predicted_answer.lower().strip() or predicted_answer.lower().strip() + 's' == correct_answer.lower().strip():
        return 0.75, [0.75] * num_evaluations
    
    scores = []
    for evaluation_attempt in range(num_evaluations):
        prompt = f"""
        Evaluate the accuracy of the predicted answer according to strict criteria below.

        Input:
        Question: {question}
        Answer type: {answer_type}
        Correct answer: {correct_answer}
        Predicted answer: {predicted_answer}

        Evaluation Rules:
        1. Return ONLY a numeric score from [1.0, 0.75, 0.5, 0.25, 0.0].
        2. Answer Type Considerations:
        - Yes/No: Must be exactly correct (1.0) or wrong (0.0)
        - Number: Must be exactly correct (1.0) or wrong (0.0)
        - Other: Focus PRIMARILY on semantic similarity:

        Scoring Criteria:
        - 1.0: Contains the correct core information, even if phrased differently or with additional details
        - 0.75: Mostly correct but missing minor information or containing slight inaccuracies
        - 0.5: Partially correct - contains some correct elements but misses important aspects
        - 0.25: Slightly correct - has a small element of the correct answer but is mostly wrong
        - 0.0: Completely incorrect, contradicts the correct answer, or avoids answering
        """
        
        for attempt in range(max_retries):
            try:
                if is_openai_model:
                    completion = client.chat.completions.create(
                        model=MODEL_NAME,
                        messages=[{"role": "user", "content": prompt}],
                        max_tokens=10,
                        temperature=0.1
                    )
                    response = completion.choices[0].message.content.strip()
                else:
                    completion = client.messages.create(
                        model=MODEL_NAME,
                        messages=[{
                            "role": "user",
                            "content": prompt
                        }],
                        max_tokens=10,
                        temperature=0.1
                    )
                    response = completion.content[0].text.strip()

                numeric_match = re.search(r'(1\.0|0\.75|0\.5|0\.25|0\.0)', response)
                if numeric_match:
                    score = float(numeric_match.group(1))
                else:
                    score = 0.0

                scores.append(score)
                break

            except Exception as e:
                print(f"Evaluation attempt {evaluation_attempt+1}, retry {attempt+1} failed: {e}")
                if attempt < max_retries - 1:
                    time.sleep(retry_delay)
                    continue
                # If all retries for this evaluation fail, continue to next evaluation

    # If all evaluations failed, return 0
    if not scores:
        return 0.0, []
        
    # Calculate the result using majority voting
    from collections import Counter
    vote_counter = Counter(scores)
    majority_score, count = vote_counter.most_common(1)[0]  # Get most common score
    
    # If there's a tie, calculate average of the tied values
    if len(scores) > 2:  # Only check for ties with more than 2 scores
        top_scores = vote_counter.most_common()
        if len(top_scores) > 1 and top_scores[0][1] == top_scores[1][1]:  # If there's a tie
            # Find all scores with the same count
            tied_scores = [score for score, count in top_scores if count == top_scores[0][1]]
            majority_score = sum(tied_scores) / len(tied_scores)  # Average of tied scores
    
    return majority_score, scores

# Evaluate model performance

In [ ]:
try:
    # Initialize counters
    correct_count = 0
    total_count = 0
    # Initialize results storage
    accuracies = []

    # Load dataset
    questions, annotations, question_id_to_answer, question_id_to_answer_type = load_dataset(annotation_path, question_path)

    if not questions:
        print("The 'questions' list is empty or not a list.")
        raise ValueError("Questions list is empty")

    # Get sample data
    sampled_questions, sampled_correct_answers = get_dataset(questions, question_id_to_answer)

    if not sampled_questions:
        print("Failed to sample questions or empty sample")
        raise ValueError("No sampled questions")

    # Create an empty collection to store the processed problem IDs
    processed_question_ids = set() 

    # Process each question
    for i, (question_item, correct_answer) in enumerate(tqdm(zip(sampled_questions, sampled_correct_answers),
                                     total=len(sampled_questions))):
        try:
            question_id = question_item['id']
            # Skip if already processed this question
            if question_id in processed_question_ids:
                continue
                
            processed_question_ids.add(question_id)
            
            question = question_item['question']
            answer_type = question_id_to_answer_type[question_id]['answer_type']
            # Use the pre-encoded image_base64
            image_base64 = question_item.get('image_base64')

            if image_base64 is None:
                print(f"Skipping question ID {question_id} due to missing image encoding")
                continue
            
            print(f"\nProcessing question {i + 1}/{len(sampled_questions)}: ID {question_id}")

            # Multi agent processing
            visual_desc = visual_agent(image_base64, question=question)
            if visual_desc is None:
                print(f"Skipping question ID {question_id} - Failed to get image description")
                continue
            
            initial_predicted_answer = language_agent(question, image_base64, visual_desc)
            if initial_predicted_answer is None:
                print(f"Skipping question ID {question_id} - Failed to generate answer")
                continue

            final_answer = hallucination_agent(
                question=question,
                initial_predicted_answer=initial_predicted_answer,
                image_base64=image_base64,
                visual_desc=visual_desc
            )

            model_answer = final_answer if final_answer else initial_predicted_answer

            # Calculate accuracy
            accuracy, scores = compute_accuracy(
                question=question,
                correct_answer=correct_answer,
                predicted_answer=model_answer,
                answer_type=answer_type
            )

            # Update counters for percentage-based accuracy
            total_count += 1
            if accuracy >= 0.75:  # Consider it correct if score is 0.75 or higher
                correct_count += 1

            # Store result
            result = {
                'question_id': question_id,
                'question': question,
                'answer_type': answer_type,
                'correct_answer': correct_answer,
                'predicted_answer': model_answer,
                'evaluator_scores': ", ".join([str(s) for s in scores]) if scores else "",  
                'accuracy': accuracy
            }
            results_standard.append(result)
            accuracies.append(accuracy)

            # Print results
            print(f"Question ID: {question_id}")
            print(f"Question: {question}")
            print(f"Answer Type: {answer_type}")
            print(f"Correct Answer: {correct_answer}")
            print(f"Predicted Answer: {model_answer}")
            if scores:
                print(f"Evaluator Scores: {scores}")
                print(f"Accuracy: {accuracy:.4f}")
            else:
                print(f"Warning: No accuracy for question: {question}")

        except Exception as e:
            print(f"Error processing question {question_item.get('id', 'unknown')}: {e}")
            continue

    # Calculate average accuracy two ways
    average_accuracy_mean = np.mean(accuracies) if accuracies else 0
    average_accuracy_ratio = correct_count / total_count if total_count > 0 else 0
    
    print(f"Average Accuracy (mean of scores): {average_accuracy_mean:.4f}")
    print(f"Average Accuracy (correct/total): {average_accuracy_ratio:.4f}")
    
    # Use the ratio method as the final average accuracy
    average_accuracy = average_accuracy_ratio

except Exception as e:
    print(f"Unexpected error: {e}")
    average_accuracy = 0

# Save results

In [ ]:
# Clean up evaluation results to remove any existing average rows
results_standard = [r for r in results_standard if r['question_id'] != 'Average']

# Ensures no duplicate summary rows when saving results
unique_questions = len(set(r['question_id'] for r in results_standard))

# Add row numbers to each result
for i, result in enumerate(results_standard, 1):
    result['row_num'] = i

# Add average accuracy as the last row
average_result = {
    'row_num': len(results_standard) + 1,
    'question_id': 'Average',
    'question': f'Total Questions: {unique_questions}',  
    'answer_type': 'All',  
    'correct_answer': '',
    'predicted_answer': '',
    'evaluator_scores': '',
    'accuracy': average_accuracy 
}
results_standard.append(average_result)

# Define column order with row_num first
column_order = [
    'row_num',
    'question_id',
    'question',
    'answer_type',
    'correct_answer',
    'predicted_answer',
    'evaluator_scores',
    'accuracy'
]

# Create safe model name for file
safe_model_name = MODEL_NAME.replace('-', '_').replace('.', '_')

# Save to CSV
results_dir = os.path.join(os.getcwd(), "results")
os.makedirs(results_dir, exist_ok=True)
# Create standard subdirectory if it doesn't exist
standard_dir = os.path.join(results_dir, "standard")
os.makedirs(standard_dir, exist_ok=True)

# Save to standard subdirectory with model name in filename
output_path = os.path.join(results_dir, "standard", f'simpsons_multi_agent_{safe_model_name}.csv')

# Check if the file exists and explicitly remove it
if os.path.exists(output_path):
    try:
        os.remove(output_path)
        print(f"Existing file removed: {output_path}")
    except Exception as e:
        print(f"Error removing existing file: {e}")

# Convert to DataFrame and save with error handling
try:
    results_df = pd.DataFrame(results_standard)
    results_df = results_df[column_order]
    
    # Save with explicit file opening to ensure it closes properly
    results_df.to_csv(output_path, index=False)
    
    # Verify the file was created
    if os.path.exists(output_path):
        print(f"Results successfully saved to: {output_path}")
    else:
        print(f"Warning: File was not created at {output_path}")
except Exception as e:
    print(f"Error saving results to CSV: {e}")

Existing file removed: /Users/wt/PythonProjects/Multi_Agent_Cartoon/results/standard/simpsons_multi_agent_claude_3_5_haiku_20241022.csv
Results successfully saved to: /Users/wt/PythonProjects/Multi_Agent_Cartoon/results/standard/simpsons_multi_agent_claude_3_5_haiku_20241022.csv
